<div style="background: linear-gradient(135deg, #0a3d2e, #145a3c, #1a7a50); padding: 40px 30px; border-radius: 16px; text-align: center; margin-bottom: 10px;">
  <h1 style="color: #d4f5e9; font-family: 'Segoe UI', sans-serif; font-size: 2.2em; margin: 0 0 8px 0;">🌍 Pipeline ETL com API REST</h1>
  <h3 style="color: #7ecba1; font-family: 'Segoe UI', sans-serif; font-weight: 300; margin: 0;">Semana 2 · Fontes de Dados, APIs, ETL vs ELT</h3>
  <p style="color: #4a9970; margin: 16px 0 0 0; font-size: 0.9em;">requests · JSON · pandas · limpeza · salvamento</p>
  <p style="color: #3a7a58; margin: 6px 0 0 0; font-size: 0.8em;">API: RestCountries v3.1 — sem chave, sem cadastro</p>
</div>

---
## 📖 Introdução

Neste notebook será realizada uma **análise exploratória de dados (EDA) com informações de países do mundo**, utilizando dados extraídos diretamente da **RestCountries API**.

O objetivo é percorrer todo o fluxo do projeto:

1. **Extração dos dados via API**
2. **Transformação e normalização do JSON**
3. **Criação de novas métricas e insights**
4. **Visualizações exploratórias**
5. **Exportação dos dados tratados**

A API utilizada é a **RestCountries v3.1**, um serviço público que fornece informações sobre países, como:

- nome
- capital
- região
- subregião
- população
- área
- idiomas
- moedas
- fusos horários
- status de independência

Endpoint principal utilizado:

`https://restcountries.com/v3.1/all`

A proposta do projeto é mostrar na prática um fluxo completo de **consumo de API REST + manipulação de dados com pandas + visualização com matplotlib/seaborn**.

---
## 📦 0. Configuração do Ambiente

In [ ]:
# ── Instalações (necessário no Colab) ─────────────────────────────────────
# !pip install requests pandas -q

import requests          # chamadas HTTP
import pandas as pd
import numpy as np
import json
import time
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

sns.set_theme(style='darkgrid')
plt.rcParams.update({
    'figure.facecolor': '#0d1f17',
    'axes.facecolor':   '#0a1a12',
    'axes.labelcolor':  '#d4f5e9',
    'xtick.color':      '#7ecba1',
    'ytick.color':      '#7ecba1',
    'text.color':       '#d4f5e9',
    'axes.titlecolor':  '#ffffff',
    'grid.color':       '#1a3d28',
    'axes.spines.top':  False,
    'axes.spines.right':False,
})

VERDE   = '#7ecba1'
AMARELO = '#f9ca24'
CORAL   = '#ff6b6b'
AZUL    = '#74b9ff'

print('✅ Ambiente configurado!')
print(f'   requests {requests.__version__} | pandas {pd.__version__}')

---
## 🔌 1. EXTRACT — Consumindo a API

### 1.1 Extração dos dados

A primeira etapa do projeto consiste em consumir a **RestCountries API**, responsável por fornecer informações estruturadas sobre países do mundo.

Nesta fase, faremos chamadas HTTP para obter dados como:

- nome
- capital
- região
- subregião
- população
- área
- idiomas
- moedas
- fusos horários
- status de independência

O endpoint principal utilizado será:

`https://restcountries.com/v3.1/all`

In [ ]:
# ── 1.2  Chamada HTTP com tratamento de erros ──────────────────────────────

URL_BASE = 'https://restcountries.com/v3.1'

# Campos que queremos (evita baixar dados desnecessários)
CAMPOS = 'name,capital,region,subregion,population,area,languages,currencies,timezones,independent'

def buscar_paises(endpoint: str = '/all', params: dict = None) -> list:
    """
    Faz uma chamada GET à API RestCountries.
    
    Parâmetros
    ----------
    endpoint : str   — rota da API (ex: '/all', '/region/americas')
    params   : dict  — query parameters opcionais
    
    Retorna
    -------
    list — dados brutos em JSON (lista de dicionários)
    """
    url = URL_BASE + endpoint
    params = params or {'fields': CAMPOS}

    print(f'🌐 Chamando: {url}')
    print(f'   Params: {params}')

    inicio = time.time()
    response = requests.get(url, params=params, timeout=15)
    elapsed = time.time() - inicio

    # ── Inspecionando a resposta ──────────────────────────────────────────
    print(f'\n📡 Status code : {response.status_code}')   # 200 = OK
    print(f'   Tempo        : {elapsed:.2f}s')
    print(f'   Tamanho      : {len(response.content) / 1024:.1f} KB')
    print(f'   Content-Type : {response.headers.get("Content-Type", "—")}')

    # ── Tratamento de erros HTTP ──────────────────────────────────────────
    # raise_for_status() lança exceção para qualquer status 4xx ou 5xx
    response.raise_for_status()

    return response.json()   # desserializa JSON → lista Python


# ── Executar a extração ───────────────────────────────────────────────────
dados_brutos = buscar_paises('/all')

print(f'\n✅ Extração concluída!')
print(f'   Total de países recebidos: {len(dados_brutos)}')
print(f'   Tipo do objeto retornado : {type(dados_brutos)}')

In [ ]:
# ── 1.3  Inspecionando o JSON bruto ───────────────────────────────────────

print('🔍 Estrutura de UM país (JSON bruto):')
print('─' * 55)

# Encontrar o Brasil no resultado
brasil = next(p for p in dados_brutos if p['name']['common'] == 'Brazil')
print(json.dumps(brasil, indent=2, ensure_ascii=False))

In [ ]:
# ── 1.4  Salvando os dados BRUTOS (etapa Load do ELT / raw do ETL) ────────

with open('countries_raw.json', 'w', encoding='utf-8') as f:
    json.dump(dados_brutos, f, ensure_ascii=False, indent=2)

print('💾 Dados brutos salvos em: countries_raw.json')
print(f'   {len(dados_brutos)} registros')

---
## 🔄 2. TRANSFORM — Limpeza e Normalização

O JSON da API tem **campos aninhados** (dicionários dentro de dicionários).  
Precisamos **achatar (flatten)** para criar um DataFrame tabular.

```
JSON bruto                          DataFrame final
──────────────────────────          ──────────────────────────────────
name → { common: 'Brazil' }    →    nome_comum
capital → ['Brasília']         →    capital
languages → { por: 'Port...' } →    idiomas (lista separada por vírgula)
currencies → { BRL: {...} }    →    moedas
population → 215313498         →    populacao
```

In [ ]:
# ── 2.1  Função de normalização (flatten) ─────────────────────────────────

def normalizar_pais(pais: dict) -> dict:
    """
    Recebe um dicionário de país (JSON bruto) e retorna
    um dicionário plano, pronto para virar linha de DataFrame.
    """
    # ── Nome ──────────────────────────────────────────────────────────────
    nome_comum   = pais.get('name', {}).get('common', None)
    nome_oficial = pais.get('name', {}).get('official', None)

    # ── Capital (lista → string) ───────────────────────────────────────────
    capitais = pais.get('capital', [])
    capital  = capitais[0] if capitais else None

    # ── Idiomas (dict → lista de valores → string) ────────────────────────
    idiomas_dict = pais.get('languages', {})
    idiomas      = ', '.join(sorted(idiomas_dict.values())) if idiomas_dict else None
    n_idiomas    = len(idiomas_dict)

    # ── Moedas (dict → primeiro código) ───────────────────────────────────
    moedas_dict  = pais.get('currencies', {})
    moedas       = ', '.join(moedas_dict.keys()) if moedas_dict else None
    nomes_moedas = ', '.join(
        m.get('name', '') for m in moedas_dict.values()
    ) if moedas_dict else None

    # ── Fusos horários ─────────────────────────────────────────────────────
    fusos   = pais.get('timezones', [])
    n_fusos = len(fusos)

    return {
        'nome_comum':     nome_comum,
        'nome_oficial':   nome_oficial,
        'capital':        capital,
        'regiao':         pais.get('region'),
        'sub_regiao':     pais.get('subregion'),
        'populacao':      pais.get('population'),
        'area_km2':       pais.get('area'),
        'idiomas':        idiomas,
        'n_idiomas':      n_idiomas,
        'moedas_codigo':  moedas,
        'moedas_nome':    nomes_moedas,
        'n_fusos':        n_fusos,
        'independente':   pais.get('independent', False),
    }


# ── Aplicar para todos os países ──────────────────────────────────────────
registros = [normalizar_pais(p) for p in dados_brutos]
df = pd.DataFrame(registros)

print(f'✅ DataFrame criado: {df.shape[0]} países × {df.shape[1]} colunas')
df.head(5)

In [ ]:
# ── 2.2  Inspeção pós-normalização ────────────────────────────────────────
print('=== info() ===')
df.info()
print()
print('=== Nulos por coluna ===')
print(df.isnull().sum().to_string())

In [ ]:
# ── 2.3  Conversão de tipos ───────────────────────────────────────────────
#
# A API retorna population e area como number, mas podem vir como float.
# Garantimos os tipos corretos explicitamente.

df['populacao']   = pd.to_numeric(df['populacao'], errors='coerce').astype('Int64')  # inteiro nullable
df['area_km2']    = pd.to_numeric(df['area_km2'],  errors='coerce').astype(float)
df['n_idiomas']   = df['n_idiomas'].astype(int)
df['n_fusos']     = df['n_fusos'].astype(int)
df['independente']= df['independente'].astype(bool)

print('✅ Tipos convertidos:')
print(df.dtypes.to_string())

In [ ]:
# ── 2.4  Padronização de campos de texto ──────────────────────────────────

# Strip de espaços e capitalização consistente
for col in ['nome_comum', 'nome_oficial', 'capital', 'regiao', 'sub_regiao']:
    df[col] = df[col].str.strip()

# Regiao e sub_regiao: title case
df['regiao']     = df['regiao'].str.title()
df['sub_regiao'] = df['sub_regiao'].str.title()

print('✅ Campos de texto padronizados')
print()
print('Regiões únicas:')
print(df['regiao'].value_counts().to_string())

In [ ]:
# ── 2.5  Tratamento de nulos ──────────────────────────────────────────────
#
# Territórios e ilhas pequenas podem não ter capital, área ou idioma oficial.
# Identificamos e documentamos — não descartamos cegamente.

sem_capital = df[df['capital'].isna()]
sem_area    = df[df['area_km2'].isna()]
sem_idioma  = df[df['idiomas'].isna()]

print(f'Países sem capital cadastrada : {len(sem_capital)}')
if len(sem_capital): print(' ', sem_capital['nome_comum'].tolist())

print(f'\nPaíses sem área cadastrada    : {len(sem_area)}')
if len(sem_area): print(' ', sem_area['nome_comum'].tolist())

print(f'\nPaíses sem idioma cadastrado  : {len(sem_idioma)}')
if len(sem_idioma): print(' ', sem_idioma['nome_comum'].tolist())

# Preencher texto nulo com marcador explícito
df['capital']    = df['capital'].fillna('N/D')
df['sub_regiao'] = df['sub_regiao'].fillna('N/D')
df['idiomas']    = df['idiomas'].fillna('N/D')
df['moedas_codigo'] = df['moedas_codigo'].fillna('N/D')
df['moedas_nome']   = df['moedas_nome'].fillna('N/D')

print(f'\n✅ Nulos restantes: {df.isnull().sum().sum()}')

In [ ]:
# ── 2.6  Feature engineering básico ──────────────────────────────────────

# Densidade populacional (hab/km²)
df['densidade_hab_km2'] = (df['populacao'] / df['area_km2']).round(2)

# Faixa populacional
def faixa_pop(pop):
    if pd.isna(pop):     return 'Desconhecida'
    if pop < 1_000_000:  return 'Micro (<1M)'
    if pop < 10_000_000: return 'Pequeno (1–10M)'
    if pop < 50_000_000: return 'Médio (10–50M)'
    if pop < 200_000_000:return 'Grande (50–200M)'
    return 'Gigante (>200M)'

df['faixa_pop'] = df['populacao'].apply(faixa_pop)

print('✅ Novas colunas criadas: densidade_hab_km2, faixa_pop')
print()
print('Distribuição por faixa:')
print(df['faixa_pop'].value_counts().to_string())

In [ ]:
# ── 2.7  Verificação final do DataFrame ───────────────────────────────────
print('=== describe() — colunas numéricas ===')
df[['populacao','area_km2','n_idiomas','n_fusos','densidade_hab_km2']].describe().round(2)

In [ ]:
# ── 2.8  Amostra do DataFrame final ───────────────────────────────────────
df.sample(8, random_state=7)

---
## 💾 3. LOAD — Salvamento dos Dados

Salvamos em dois formatos:
- **CSV** → fácil de abrir em Excel, pandas, qualquer ferramenta
- **JSON** → preserva a estrutura tipada, ideal para APIs e bancos NoSQL

In [ ]:
# ── 3.1  Salvar CSV ───────────────────────────────────────────────────────
df.to_csv('countries_clean.csv', index=False, encoding='utf-8-sig')  # utf-8-sig = Excel-friendly

# ── 3.2  Salvar JSON (orientação records = lista de objetos) ──────────────
df.to_json('countries_clean.json', orient='records', force_ascii=False, indent=2)

print('💾 Arquivos salvos:')
print('   ├── countries_raw.json   ← dados brutos da API (sem modificação)')
print('   ├── countries_clean.csv  ← DataFrame limpo, Excel-friendly')
print('   └── countries_clean.json ← DataFrame limpo em JSON')
print()
print(f'   Shape final: {df.shape[0]} países × {df.shape[1]} colunas')

# ── 3.3  Verificação de leitura (round-trip) ──────────────────────────────
df_check = pd.read_csv('countries_clean.csv')
assert df_check.shape == df.shape, '❌ Shape diferente após salvar/ler!'
print('\n✅ Round-trip OK — CSV salvo e relido com shape idêntico')

---
## 📊 4. Análise Exploratória Rápida

In [ ]:
# ── 4.1  Países por região ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Barras: contagem por região
ax = axes[0]
contagem = df['regiao'].value_counts()
bars = ax.barh(contagem.index, contagem.values, color=VERDE, edgecolor='none', height=0.6)
for bar, val in zip(bars, contagem.values):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2,
            str(val), va='center', color='#d4f5e9', fontsize=10)
ax.set_xlabel('Quantidade de países')
ax.set_title('Países por Região', fontweight='bold')

# Pizza: % de países independentes
ax = axes[1]
independencia = df['independente'].value_counts()
ax.pie(independencia, labels=['Países Independentes', 'Territórios Não Autônomos'],
       colors=[VERDE, CORAL], autopct='%1.1f%%', startangle=90,
       textprops={'color': '#d4f5e9', 'fontsize': 11},
       wedgeprops={'edgecolor': '#0a1a12', 'linewidth': 2})
ax.set_title('Países por Independência', fontweight='bold')

fig.suptitle('Visão Geral — Países do Mundo', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 4.2  Top 15 países mais populosos ────────────────────────────────────
top15_pop = df.nlargest(15, 'populacao')[['nome_comum', 'populacao', 'regiao']]

fig, ax = plt.subplots(figsize=(12, 6))
cores = [VERDE if r == 'Americas' else AZUL if r == 'Asia' 
         else AMARELO if r == 'Africa' else CORAL
         for r in top15_pop['regiao']]

bars = ax.barh(top15_pop['nome_comum'], top15_pop['populacao'] / 1e9,
               color=cores, edgecolor='none', height=0.65)
for bar, val in zip(bars, top15_pop['populacao']):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val/1e6:.0f}M', va='center', fontsize=9, color='#d4f5e9')

ax.invert_yaxis()
ax.set_xlabel('População (bilhões)')
ax.set_title('Top 15 Países Mais Populosos', fontweight='bold', fontsize=13)

legenda = [
    mpatches.Patch(color=VERDE,   label='Americas'),
    mpatches.Patch(color=AZUL,    label='Asia'),
    mpatches.Patch(color=AMARELO, label='Africa'),
    mpatches.Patch(color=CORAL,   label='Europe/Oceania'),
]
ax.legend(handles=legenda, fontsize=9, loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# ── 4.3  Dispersão: área vs população por região ──────────────────────────
regiao_cores = {
    'Americas': VERDE, 'Asia': AZUL, 'Africa': AMARELO,
    'Europe': CORAL, 'Oceania': '#fd79a8', 'Antarctic': '#b2bec3'
}

df_plot = df.dropna(subset=['area_km2', 'populacao'])
df_plot = df_plot[df_plot['area_km2'] > 0]

fig, ax = plt.subplots(figsize=(12, 6))

for regiao, grupo in df_plot.groupby('regiao'):
    ax.scatter(
        grupo['area_km2'], grupo['populacao'],
        c=regiao_cores.get(regiao, '#dfe6e9'),
        label=regiao, alpha=0.7, s=40, edgecolors='none'
    )

# ── Países notáveis ────────────────────────────────────────────────────────
paises_notaveis = {
    'Brazil': 'Brasil',
    'Bangladesh': 'Bangladesh',
    'Mongolia': 'Mongólia',
    'Vatican City': 'Vaticano',
    'Greenland': 'Groenlândia',
    'Hong Kong': 'Hong Kong',
}

for pais_api, label_pt in paises_notaveis.items():
    pais = df_plot[df_plot['nome_comum'] == pais_api]

    if not pais.empty:
        row = pais.iloc[0]

        ax.scatter(
            row['area_km2'],
            row['populacao'],
            s=90,
            c='white',
            edgecolors='black',
            linewidths=1.2,
            zorder=5
        )

        ax.annotate(
            label_pt,
            xy=(row['area_km2'], row['populacao']),
            xytext=(row['area_km2'] * 1.15, row['populacao'] * 1.2),
            color='#d4f5e9',
            fontsize=10,
            fontweight='bold',
            arrowprops={'arrowstyle': '->'}
        )
        
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Área km² (escala log)')
ax.set_ylabel('População (escala log)')
ax.set_title('Área vs População por Região', fontweight='bold', fontsize=13)
ax.legend(fontsize=9, loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# ── 4.4  Média de idiomas por região ──────────────────────────────────────
media_idiomas = df.groupby('regiao')['n_idiomas'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(media_idiomas.index, media_idiomas.values,
              color=AZUL, edgecolor='none', width=0.55)
for bar, val in zip(bars, media_idiomas.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.1f}', ha='center', va='bottom', fontsize=10, color='#d4f5e9')
ax.set_ylabel('Média de idiomas oficiais')
ax.set_title('Diversidade Linguística por Região', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

---
## 🔎 5. Consultas e Filtros Úteis

In [ ]:
# ── 5.1  Países da América do Sul ─────────────────────────────────────────
am_sul = df[df['sub_regiao'] == 'South America'][['nome_comum','capital','populacao','area_km2']]
print(f'América do Sul: {len(am_sul)} países')
am_sul.sort_values('populacao', ascending=False)

In [ ]:
# ── 5.2  Países com mais de 3 fusos horários ──────────────────────────────
multi_fuso = df[df['n_fusos'] > 3][['nome_comum', 'n_fusos', 'regiao']].sort_values('n_fusos', ascending=False)
print(f'Países com > 3 fusos: {len(multi_fuso)}')
print(multi_fuso.to_string(index=False))

In [ ]:
# ── 5.3  Os 5 países mais densos e menos densos ───────────────────────────
df_dens = df.dropna(subset=['densidade_hab_km2']).query('area_km2 > 100')  # exclui microterritórios

print('🏙️  TOP 5 mais densos (hab/km²):')
print(df_dens.nlargest(5, 'densidade_hab_km2')
             [['nome_comum', 'densidade_hab_km2', 'area_km2']].to_string(index=False))

print('\n🏜️  TOP 5 menos densos (hab/km²):')
print(df_dens.nsmallest(5, 'densidade_hab_km2')
             [['nome_comum', 'densidade_hab_km2', 'area_km2']].to_string(index=False))

In [ ]:
# ── 5.4  Buscando um país específico (simula rota /name/{pais}) ───────────

def buscar_por_nome(nome: str) -> pd.Series:
    """Retorna dados de um país pelo nome comum (case-insensitive)."""
    resultado = df[df['nome_comum'].str.lower() == nome.lower()]
    if resultado.empty:
        print(f'❌ País "{nome}" não encontrado.')
        return None
    return resultado.iloc[0]

brasil = buscar_por_nome('Brazil')
print('🇧🇷 Brasil:')
print(brasil.to_string())

---
## 📋 6. Resumo do Pipeline ETL

```
┌─────────────────────────────────────────────────────────────────┐
│                     PIPELINE ETL COMPLETO                       │
├──────────────┬──────────────────────────┬───────────────────────┤
│   EXTRACT    │       TRANSFORM          │        LOAD           │
├──────────────┼──────────────────────────┼───────────────────────┤
│ requests.get │ Flatten JSON aninhado    │ .to_csv()             │
│ status_code  │ Conversão de tipos       │ .to_json()            │
│ .json()      │ Padronização de texto    │ countries_clean.csv   │
│              │ Tratamento de nulos      │ countries_clean.json  │
│ → raw.json   │ Feature engineering      │                       │
└──────────────┴──────────────────────────┴───────────────────────┘

Fonte  : RestCountries v3.1 API
Input  : ~250 países em JSON aninhado
Output : DataFrame 250 × 17 colunas, limpo e tipado
```

### Conceitos praticados

| Conceito | Onde apareceu |
|---|---|
| **API REST / GET** | `requests.get()` com URL e params |
| **Status code** | `response.status_code`, `raise_for_status()` |
| **JSON → Python** | `response.json()` |
| **Flatten de JSON aninhado** | Função `normalizar_pais()` |
| **Conversão de tipos** | `pd.to_numeric()`, `.astype()` |
| **Padronização de texto** | `.str.strip()`, `.str.title()` |
| **Tratamento de nulos** | `.fillna()`, análise por coluna |
| **Salvamento CSV / JSON** | `.to_csv()`, `.to_json()` |
| **ETL** | Extract → Transform → Load em sequência |